# HAC Steel OCBF — Base Fuse Force (Lateral & Longitudinal) · **v02**

**PBD rocking-fuse options study — ASCE 7-22 (Ch. 12 & 15), AISC 360-22 / 341-22, 2025 CBC**

| field | value |
|---|---|
| Project | OAI Richmond Robotic Lab — Phase 2 |
| Job No. | DG26.0160.00 |
| Location | 1411 Harbour Way S., Richmond, CA |
| Date | 2026-08-03 |
| Subject | HAC base fuse force — lateral kicker fuses & longitudinal braced-bay fuse |
| Prepared | Jeffrey Dai (Structural Intern) |
| Checked | S. Aher (pending) |

**Basis.** 2025 CBC / ASCE 7-22 (Ch. 12 & 15); AISC 360-22 & 341-22. The permit design stays code-based (ELF, Steel OCBF, `R = 3.25`, `Cs = 0.402`, RC II / `Ie = 1.0`); this sheet supports the beyond-code PBD rocking-fuse options study. **Lengths and moments are carried in ft and kip·ft; forces in kip.**

**Purpose.** Size the single ductile threaded-rod base fuse in each principal direction from hand statics: (1) the **lateral (transverse) kicker fuses**, and (2) the **longitudinal braced-bay fuse**. The fuse is a pure axial element inline with each brace; it yields in tension and caps the force delivered to the capacity-protected base plate, `¾″⌀ Gr. 105` anchors, gusset, and reinforced pier (S9.00).

**Changes from v01**
- All lengths and moments standardized to **ft / kip·ft** (v01 rendered kip-in).
- Split into distinct **Lateral** and **Longitudinal** sections.
- **Removed** the RISA *governing base-end forces* section; fuse demands now come from hand statics in each direction.
- Lateral direction rebuilt to the as-drawn geometry: **7 bays (5 interior + 2 exterior), 8 bents, 16 kickers (2 per bent), one fuse per kicker**, with a tributary load distribution.

> **PRELIMINARY — PBD OPTIONS STUDY — NOT FOR PERMIT / CONSTRUCTION**

> **Key inputs.** Whole-HAC RISA dead `W ≈ 230 kip`. Transverse bent: columns 8 ft apart, kicker feet 14.5 ft apart (3′-3″ outboard of each column, S3.04/1); kicker rise ≈ 7.5 ft to the work point — **confirm against S3.04**. Longitudinal: central braced bay 9′-8″ wide (S3.03). Single-rod fuse concept and `0.7h` overturning simplification per call with S. Aher.

In [1]:
%%capture
# --- Environment setup -------------------------------------------------
!pip install -q handcalcs forallpeople numpy pandas

import sys, os
# Make the project root (the folder containing 'helpers') importable.
for cand in [os.getcwd(), os.path.abspath('..'), os.path.abspath(os.path.join('..', '..'))]:
    if os.path.isdir(os.path.join(cand, 'helpers')) and cand not in sys.path:
        sys.path.insert(0, cand)
        break

# One import brings: units (kip, ft, ksi, ...), math (sin, cos, atan, sqrt, pi, ...),
# sig(), mag(), mag_ft(), export_notebook(), setup_formatting(), and the %%render magic.
from helpers.formatting import *
setup_formatting(3, system="kip-ft")   # 3 sig figs; lengths in ft, moments in kip*ft

## 1  Seismic Design Parameters (ASCE 7-22 / 2025 CBC)

Design spectral values from the Concept BOD (Site Class D, SDC D). $S_{DS}$ sets the short-period plateau; $\Omega_0$ and $I_e$ per Table 15.4-1 for a Steel OCBF designed as a nonbuilding structure similar to a building. $S_{DS} = 1.307$ g gives the locked $C_s = 0.402$.

In [2]:
%%render params
S_S     = 1.960          # g, mapped $MCE_R$ (ASCE 7-22 Ch. 22)
S_1     = 0.680          # g, mapped $MCE_R$
S_DS    = 1.307          # g, design short-period spectral accel. (governs)
S_D1    = 1.133          # g, design 1-second spectral accel.
R       = 3.250          # Steel OCBF response modification factor (Table 15.4-1)
Omega_0 = 2.000          # overstrength factor for capacity-protected elements
I_e     = 1.000          # importance factor, Risk Category II
T_a     = 0.182          # s, approximate fundamental period (Eq. 12.8-7)
W       = 230*kip        # whole-HAC seismic weight (RISA dead)
h       = 19.48*ft       # top-of-steel height (S3.04/1)

<IPython.core.display.Latex object>

## 2  Whole-Structure Base Shear (ELF, ASCE 7-22 §12.8)

Whole-HAC weight with the locked $C_s$. $V$ is the reduced design base shear; $V_e = V(R/I_e) = S_{DS} W$ is the elastic demand with no ductility. The gap between them is what the fuse ductility must absorb. The same magnitude applies in each principal direction.

In [3]:
%%render
C_s = S_DS / (R / I_e)          # Eq. 12.8-2 seismic response coefficient (locked at 0.402)
V   = C_s * W                   # Eq. 12.8-1 design base shear, each principal direction
V_e = V * (R / I_e)             # elastic (unreduced) demand = $S_{DS} \cdot W$

<IPython.core.display.Latex object>

With $T_a = 0.182$ s well below $T_s$, the HAC sits on the short-period plateau and draws near-peak spectral acceleration — the physical root of the server-acceleration problem the fuse addresses.

## 3  Vertical Seismic Load Effect

Reduces the gravity available for overturning restoring; governs the uplift combination.

In [4]:
%%render
E_v = 0.2 * S_DS          # Eq. 12.4-4a vertical seismic effect (fraction of dead load)

<IPython.core.display.Latex object>

## 4  Overturning Demand (shared by both directions)

Apply the base shear $V$ as a single resultant at $0.7h$ (S. Aher simplification; use full $h$ for a conservative bound). $M_{ot}$ is the same in each direction — it feeds the lateral stability check and the longitudinal braced-bay bracket.

In [5]:
%%render params
k_h = 0.700              # equivalent-force height factor ($0.7h$ per S. Aher)

<IPython.core.display.Latex object>

In [6]:
%%render
h_eff = k_h * h                 # effective height of the lateral resultant
M_ot  = V * h_eff               # base overturning moment, each principal direction

<IPython.core.display.Latex object>

$M_{ot} \approx 1{,}260$ kip·ft at $0.7h$. Balanced against self-weight restoring (lateral) and resolved into the braced-bay couple (longitudinal) below.

# Lateral (Transverse) Direction

## 5  Geometry &amp; Load Distribution

The transverse system is a set of **base kickers** — no upper diaphragm or X-brace. Each of the **8 bents** (column lines 1-1 → 1-8) has **2 kickers**, one per column, splayed outboard from the 8 ft column line to feet 14.5 ft apart. That is **16 kickers total (8 per side), one ductile fuse per kicker**.

Under a transverse push each bent develops one windward (tension) kicker and one leeward (compression) kicker; their horizontal components share the bent shear. With no diaphragm, each bent resists its own tributary inertia, so the bent over the widest tributary (adjacent to an end bay) governs.

In [7]:
%%render params
n_bent    = 8              # transverse bents (column lines 1-1 through 1-8)
kick_bent = 2              # kickers per bent (one per column, splayed outboard)
b_foot    = 7.25*ft        # half of the kicker-foot spread (feet 14.5 ft apart)
run_k     = 3.25*ft        # kicker horizontal run, foot to column (S3.04/1)
rise_k    = 9.25*ft         # kicker vertical rise to the work point (S3.04/1)
w_end     = 16.625*ft      # end-bay width (2 exterior bays)
w_int     = 9.667*ft       # interior-bay width (5 interior bays)

<IPython.core.display.Latex object>

In [8]:
%%render
n_kick  = n_bent * kick_bent            # total kickers = total lateral fuses
L_tot   = 2*w_end + 5*w_int             # overall HAC length (7 bays)
w_trib  = (w_end + w_int) / 2           # governing bent tributary width (bents 1-2, 1-7)
theta_k = atan(run_k / rise_k)          # kicker angle from vertical (rad)
L_k     = rise_k / cos(theta_k)         # kicker work-point length

<IPython.core.display.Latex object>

Governing bent tributary width $=(w_{end}+w_{int})/2 = 13.15$ ft (bents 1-2, 1-7). Kicker angle $\theta_k \approx 23°$ from vertical. Column bases taken as pinned, so the kickers carry the full transverse base shear at each bent.

In [9]:
%%render
V_lat  = C_s * W                        # transverse base shear (whole HAC)
V_bent = V_lat * (w_trib / L_tot)       # shear at the governing bent (tributary, no diaphragm)
P_k    = V_bent / (2 * sin(theta_k))    # kicker axial = lateral fuse demand (windward tension)

<IPython.core.display.Latex object>

## 6  Overturning Stability (global)

Whole-structure transverse overturning about a foot line, resisted by self-weight at the 7.25 ft lever. This confirms the structure does not tip as a rigid body; the kicker axial above governs the fuse.

In [10]:
%%render
M_rest  = W * b_foot                     # self-weight restoring at full dead (1.0D)
SR      = M_rest / M_ot                  # global stability ratio ($>1$ -> no rigid-body tipping)
W_up    = (0.9 - E_v) * W                # gravity in the uplift combination ($0.9D - E_v$)
M_netup = M_ot - W_up * b_foot           # net overturning beyond restoring
T_up    = M_netup / (2 * b_foot)         # total windward-line uplift (global; small)

<IPython.core.display.Latex object>

$SR \approx 1.3 > 1$ — globally stable. The net rigid-body uplift $T_{up}$ spread over 8 windward feet is negligible per foot, so the **kicker axial $P_k \approx 19$ kip governs the lateral fuse**, not global overturning.

## 7  Lateral Fuse Sizing

Single ductile reduced-shank threaded rod inline with each kicker, sized to the kicker axial tension: reduced-shank yield $\phi_t F_y A \ge T_u$.

*Unit convention: structural lengths and moments are in ft / kip·ft; rod strength is in ksi and rod diameters in inches, per steel-specification practice (rods are ordered by inch diameter).*

In [11]:
# Lateral fuse demand carried forward as a plain kip magnitude for rod sizing.
T_u_lat = float(mag(P_k))    # kip

In [12]:
%%render params
F_yr  = 105          # ksi, reduced-shank yield (F1554 Gr. 105)
phi_t = 0.900        # tension-yield resistance factor

<IPython.core.display.Latex object>

In [13]:
%%render
A_lat = T_u_lat / (phi_t * F_yr)     # reduced-shank area, lateral fuse [$\mathrm{in}^2$]
D_lat = sqrt(4 * A_lat / pi)         # reduced-shank diameter, lateral fuse [in]

<IPython.core.display.Latex object>

$D_{lat} \approx 0.50$ in → **⅝″⌀ reduced shank** ($A = 0.307\ \mathrm{in}^2$, $\phi F_y A \approx 29\ \mathrm{kip} \ge 19\ \mathrm{kip}$): stays elastic at the code demand and yields below the MCE-elastic force — the intended fuse behavior. The capacity-protected base plate, anchors, gusset, and pier are designed for the amplified force $\Omega_0 P_k \approx 37$ kip (or the fuse expected overstrength $R_y R_t \phi F_y A$), in the Week-2 50% package.

# Longitudinal Direction

## 8  System Behavior

The longitudinal line is a stiffness hybrid: one central chevron-braced bay carries the primary lateral resistance while the other bays are welded HSS moment frames, tied to a common drift by continuous columns and beams. The braced bay draws the large majority of story shear; with no slab, the level beams act as collectors dragging inertia into that bay. Overturning therefore concentrates as a tension/compression couple in the braced-bay columns — the base fuse location. A conservative single-bay bracket (all $V$ to one bay, no moment-frame sharing) sizes the fuse; RISA / Perform-3D refine it downward.

In [14]:
%%render params
B_bay  = 9.667*ft        # central braced-bay width (overturning lever, S3.03)
P_colD = 13.5*kip        # gravity pin-down at the braced-bay column (dead, per BOD)

<IPython.core.display.Latex object>

In [15]:
%%render
V_lon  = C_s * W                     # longitudinal base shear (whole HAC)
T_hand = M_ot / B_bay - P_colD       # braced-bay overturning bracket = longitudinal fuse demand

<IPython.core.display.Latex object>

## 9  Longitudinal Fuse Sizing

Same reduced-shank rod inline with the braced-bay base diagonal (S9.00/5), sized to the base-end tension. Rod strength in ksi and diameter in inches, per the convention noted in §7.

In [16]:
# Longitudinal fuse demand carried forward as a plain kip magnitude for rod sizing.
T_u_lon = float(mag(T_hand))    # kip

In [17]:
%%render
A_lon = T_u_lon / (phi_t * F_yr)     # reduced-shank area, longitudinal fuse [$\mathrm{in}^2$]
D_lon = sqrt(4 * A_lon / pi)         # reduced-shank diameter, longitudinal fuse [in]

<IPython.core.display.Latex object>

$D_{lon} \approx 1.26$ in → **1¼″⌀ reduced shank** ($A = 1.227\ \mathrm{in}^2$, $\phi F_y A \approx 116\ \mathrm{kip}$). $T_{hand}$ is a conservative bracket, so the true demand is lower; the capacity-protected hardware carries $\Omega_0 T_{hand}$ (or fuse overstrength), developed in the 50% package.

## 10  Results Summary

In [18]:
import pandas as pd
q = lambda x: sig(mag_ft(x))                       # kip-ft-system magnitude (kip or kip*ft)
rows = [
    ("Whole-HAC base shear  V",        f"{q(V)} kip",        "Cs*W, R = 3.25"),
    ("Elastic base shear  Ve",         f"{q(V_e)} kip",      "V*(R/Ie) = SDS*W"),
    ("Overturning moment  Mot",        f"{q(M_ot)} kip-ft",  "resultant at 0.7h"),
    ("Global stability  SR",           f"{q(SR)} x",         "> 1 -> stable"),
    ("--- LATERAL (transverse) ---",   "",                   ""),
    ("Kickers = lateral fuses",        f"{int(mag_ft(n_kick))}", "8 per side, 1 fuse each"),
    ("Governing bent shear  Vbent",    f"{q(V_bent)} kip",   "tributary, no diaphragm"),
    ("Kicker axial (fuse)  Pk",        f"{q(P_k)} kip",      "windward tension"),
    ("Lateral fuse dia.  Dlat",        f"{sig(D_lat)} in",   "select 5/8 in dia"),
    ("--- LONGITUDINAL ---",           "",                   ""),
    ("Braced-bay fuse  Thand",         f"{q(T_hand)} kip",   "overturning bracket (cons.)"),
    ("Longitudinal fuse dia.  Dlon",   f"{sig(D_lon)} in",   "select 1-1/4 in dia"),
]
df = pd.DataFrame(rows, columns=["Quantity", "Value", "Note"])
df

,Quantity,Value,Note
0,Whole-HAC base shear V,92.5 kip,"Cs*W, R = 3.25"
1,Elastic base shear Ve,301 kip,V*(R/Ie) = SDS*W
2,Overturning moment Mot,1260 kip-ft,resultant at 0.7h
3,Global stability SR,1.32 x,> 1 -> stable
4,--- LATERAL (transverse) ---,,
5,Kickers = lateral fuses,16,"8 per side, 1 fuse each"
6,Governing bent shear Vbent,14.9 kip,"tributary, no diaphragm"
7,Kicker axial (fuse) Pk,22.5 kip,windward tension
8,Lateral fuse dia. Dlat,0.55 in,select 5/8 in dia
9,--- LONGITUDINAL ---,,


## 11  Reconciliation Targets for the 2D Perform-3D Model

1. Elastic period $T_1 \approx 0.18$ s (modal check).
2. First yield in the base fuse at the hand demand — lateral kicker $\approx 19$ kip, longitudinal braced-bay $\approx 117$ kip.
3. Pushover plateaus flat (elastic–plastic axial fuse); force ceiling at the fuse overstrength $R_y R_t \phi F_y A$ so the base plate, anchors, gusset, and pier stay elastic (capacity-design hierarchy).
4. Global overturning stable ($SR \approx 1.3$ at full dead); transverse net rigid-body uplift small.
5. Lateral shear distributes across the 16 kickers by tributary; longitudinal shear concentrates in the braced bay, with the moment frames picking up share after the fuse yields.

## 12  References &amp; Export

**Codes / standards.** 2025 CBC; ASCE 7-22 §12.8, §12.4, Table 15.4-1; AISC 360-22 §E3; AISC 341-22 §F1 (OCBF); ACI 318-19 Ch. 17 (anchorage, Week-2).
**Geometry.** Transverse bent and base kickers per **S3.04/1**; longitudinal braced bay per **S3.03**; base details **S9.00/1** (short/transverse brace) and **S9.00/5** (long/longitudinal brace).
**Basis.** Whole-structure weight and locked $C_s$ per the Concept BOD; single-rod fuse concept and $0.7h$ overturning simplification per call with S. Aher. Companion to *OCBF_BaseShear_CapacityReconciliation*. Prepared with `handcalcs` + `forallpeople`.

In [19]:
# export_notebook("html")   # run after executing to write outputs/ (off during batch execution)